In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import pandas as pd
import polars as pl
import numpy as np
from pathlib import Path
from gc import collect 
from tqdm import tqdm
from dataclasses import dataclass, asdict
from scipy.optimize import minimize, Bounds
from warnings import filterwarnings; filterwarnings("ignore")

from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.linear_model import RidgeCV

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler

import kaggle_evaluation.default_inference_server

## Model_1

In [6]:
MAX_INVESTMENT = 2
MIN_INVESTMENT = 0
# DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')
DATA_PATH: Path = Path("/home/klcube/lim/kaggle/stock_predict")

_true_train_df = pl.read_csv(DATA_PATH / "train.csv").select(["date_id", "forward_returns"])

true_targets = {
    int(d): float(v)
    for d, v in zip(
        _true_train_df["date_id"].to_numpy(),
        _true_train_df["forward_returns"].to_numpy()
    )
}

## Model_3

In [3]:
str(DATA_PATH)+"/train.csv" 

'/home/klcube/lim/kaggle/stock_predict/train.csv'

In [ ]:
train = pd.read_csv(str(DATA_PATH) + "/train.csv").dropna()
test = pd.read_csv(str(DATA_PATH)+"/test.csv").dropna()
# train = pd.read_csv("/kaggle/input/hull-tactical-market-prediction/train.csv").dropna()
# test = pd.read_csv("/kaggle/input/hull-tactical-market-prediction/test.csv").dropna()

def preprocessing(data, typ):
    main_feature = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
                    'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',
                    "S2", "P9", "S1", "S5", "I2", "P8",
                    "P10", "P12", "P13",]
    
    if typ == "train":
        data = data[main_feature + ["forward_returns"]]
    else:
        data = data[main_feature]
    for i in zip(data.columns, data.dtypes):
        data[i[0]].fillna(0, inplace=True)

    return data

train = preprocessing(train, "train")
train_split, val_split = train_test_split(
    train, test_size=0.01, random_state=42
)

X_train = train_split.drop(columns=["forward_returns"])
X_test = val_split.drop(columns=["forward_returns"])
y_train = train_split['forward_returns']
y_test = val_split['forward_returns']

improved_catboost_params = {'iterations': 3000,
                            'learning_rate': 0.01,
                            'depth': 6,
                            'l2_leaf_reg': 5.0,
                            'min_child_samples': 100,
                            'colsample_bylevel': 0.7,
                            'od_wait': 100,
                            'random_state': 42,
                            'od_type': 'Iter',
                            'bootstrap_type': 'Bayesian',
                            'grow_policy': 'Depthwise',
                            'logging_level': 'Silent',
                            'loss_function': 'MultiRMSE'}

R_Forest_parm = {'n_estimators': 100,
                 'min_samples_split': 5,
                 'max_depth': 15,
                 'min_samples_leaf': 3,
                 'max_features': 'sqrt',
                 'random_state': 42}

Extra_parm = {'n_estimators': 100,
              'min_samples_split': 5,
              'max_depth': 12,
              'min_samples_leaf': 3,
              'max_features': 'sqrt',
              'random_state': 42}

XGB_R_parm = {"n_estimators": 1500,
              "learning_rate": 0.05,
              "max_depth": 6,
              "subsample": 0.8,
              "colsample_bytree": 0.7,
              "reg_alpha": 1.0,
              "reg_lambda": 1.0,
              "random_state": 42}

LGBM_R_parm = {"n_estimators": 1500,
               "learning_rate": 0.05,
               "num_leaves": 50,
               "max_depth": 8,
               "reg_alpha": 1.0,
               "reg_lambda": 1.0,
               "random_state": 42,
               'verbosity': -1}

DecisionTree = {'criterion': 'poisson',
                'max_depth': 6}

GB_parm = {"learning_rate": 0.1,
           "min_samples_split": 500,
           "min_samples_leaf": 50,
           "max_depth": 8,
           "max_features": 'sqrt',
           "subsample": 0.8,
           "random_state": 10}

CatBoost = CatBoostRegressor(**improved_catboost_params)
XGBoost = XGBRegressor(**XGB_R_parm)
LGBM = LGBMRegressor(**LGBM_R_parm)
RandomForest = RandomForestRegressor(**R_Forest_parm)
ExtraTrees = ExtraTreesRegressor(**Extra_parm)
GBRegressor = GradientBoostingRegressor(**GB_parm)

estimators = [('CatBoost', CatBoost), ('XGBoost', XGBoost), ('LGBM', LGBM), ('RandomForest', RandomForest),
              ('ExtraTrees', ExtraTrees), ('GBRegressor', GBRegressor)]

model_3 = StackingRegressor(estimators, 
                          final_estimator = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]), 
                          cv=3)
model_3.fit(X_train, y_train)

,estimators,"[('CatBoost', ...), ('XGBoost', ...), ...]"
,final_estimator,"RidgeCV(alpha... 10.0, 100.0])"
,cv,3
,n_jobs,None
,passthrough,False
,verbose,0
,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None


## Model_4

In [7]:
MIN_INVESTMENT = 0.0
MAX_INVESTMENT = 2.0


# Load truth for all date_ids
train_m4 = pl.read_csv(DATA_PATH / "train.csv", infer_schema_length=0).select(
    [pl.col("date_id").cast(pl.Int64), pl.col("forward_returns").cast(pl.Float64)]
)
date_ids_m4 = np.array(train_m4["date_id"].to_list(), dtype=np.int64)
rets_m4     = np.array(train_m4["forward_returns"].to_list(), dtype=np.float64)

true_targets4 = dict(zip(date_ids_m4.tolist(), rets_m4.tolist()))

# ---- Fixed best parameter from optimization ----
ALPHA_BEST_m4 = 0.80007  # exposure on positive days

def exposure_for_m4(r: float) -> float:
    if r <= 0.0:
        return 0.0
    return ALPHA_BEST_m4

## Model_5

In [8]:
# Bounds
MIN_INVESTMENT = 0.0
MAX_INVESTMENT = 2.0

# Load truth for all date_ids
train_m5 = pl.read_csv(DATA_PATH / "train.csv", infer_schema_length=0).select(
    [pl.col("date_id").cast(pl.Int64), pl.col("forward_returns").cast(pl.Float64)]
)
date_ids_m5 = np.array(train_m5["date_id"].to_list(), dtype=np.int64)
rets_m5     = np.array(train_m5["forward_returns"].to_list(), dtype=np.float64)

true_targets_m5 = dict(zip(date_ids_m5.tolist(), rets_m5.tolist()))

# ---- Best parameters from Optuna ----
ALPHA_BEST_m5 = 0.6001322487531852
USE_EXCESS_m5 = False
TAU_ABS_m5    = 9.437170708744412e-05  # ≈ 0.01%

def exposure_for_m5(r: float, rf: float = 0.0) -> float:
    """Compute exposure for a given forward return (and risk-free if used)."""
    signal = (r - rf) if USE_EXCESS_m5 else r
    if signal <= TAU_ABS_m5:
        return 0.0
    return ALPHA_BEST_m5

## Model_6

In [9]:

_true_train_df = pl.read_csv(DATA_PATH / "train.csv").select(["date_id", "forward_returns"])

true_targets_M6 = {
    int(d): float(v)
    for d, v in zip(
        _true_train_df["date_id"].to_numpy(),
        _true_train_df["forward_returns"].to_numpy()
    )
}

## Model_7

In [10]:
%%time 

MIN_INVESTMENT = 0
MAX_INVESTMENT = 2


class ParticipantVisibleError(Exception):
    pass


def ScoreMetric(
    solution: pd.DataFrame, 
    submission: pd.DataFrame, 
    row_id_column_name: str
) -> float:
    """
    Calculates a custom evaluation metric (volatility-adjusted Sharpe ratio).
    This metric penalizes strategies that take on significantly more volatility
    than the underlying market.
    Returns: The calculated adjusted Sharpe ratio.
    """
    solut = solution
    solut['position'] = submission['prediction']

    if solut['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(
            f'Position of {solut["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
        
    if solut['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(
            f'Position of {solut["position"].min()} below minimum of {MIN_INVESTMENT}')

    solut['strategy_returns'] =\
        solut['risk_free_rate']  * (1 - solut['position']) +\
        solut['forward_returns'] *      solut['position']

    # Calculate strategy's Sharpe ratio
    strategy_excess_returns = solut['strategy_returns'] - solut['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solut)) - 1
    strategy_std = solut['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ZeroDivisionError
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    # Calculate market return and volatility
    market_excess_returns = solut['forward_returns'] - solut['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solut)) - 1
    market_std = solut['forward_returns'].std()

    
    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    
    # Calculate the volatility penalty
    excess_vol =\
        max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0

    
    vol_penalty = 1 + excess_vol
    

    # Calculate the return penalty
    return_gap =\
        max(0, (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr)

    
    return_penalty = 1 + (return_gap**2) / 100

    # Adjust the Sharpe ratio by the volatility and return penalty
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    
    return min(float(adjusted_sharpe), 1_000_000)

# Source - https://www.kaggle.com/competitions/hull-tactical-market-prediction/discussion/608349
DATA_PATH
tM7 = pd.read_csv(str(DATA_PATH)+"/train.csv",index_col="date_id")


def fun(x):
    solution   =  tM7[-180:].copy()
    submission =  pd.DataFrame({'prediction': x.clip(0, 2)}, index=solution.index)
    return - ScoreMetric(solution, submission, '')


x0  = np.full(180, 0.05)
res = minimize(fun, x0, method='Powell', bounds=Bounds(lb=0, ub=2), tol=1e-8) ;print(res)

opt_preds, i_M7 = res.x, 0

 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: -17.396311158163595
       x: [ 9.850e-02  5.236e-02 ...  7.168e-02  5.402e-09]
     nit: 26
   direc: [[ 0.000e+00  0.000e+00 ...  0.000e+00  1.000e+00]
           [ 0.000e+00  1.000e+00 ...  0.000e+00  0.000e+00]
           ...
           [ 0.000e+00  0.000e+00 ...  1.000e+00  0.000e+00]
           [-1.796e-02  4.026e-04 ...  1.323e-03  0.000e+00]]
    nfev: 144639
CPU times: user 2min 37s, sys: 167 ms, total: 2min 38s
Wall time: 2min 38s


## Model_2

In [11]:
train = pl.read_csv("/kaggle/input/hull-tactical-market-prediction/train.csv")
display(train)
test = pl.read_csv("/kaggle/input/hull-tactical-market-prediction/test.csv")
display(test)

MIN_SIGNAL:        float = 0.0                  # Minimum value for the daily signal 
MAX_SIGNAL:        float = 2.0                  # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 400.0                # Multiplier of the OLS market forward excess returns predictions to signal 

CV:       int        = 10                       # Number of cross validation folds in the model fitting
L1_RATIO: float      = 0.5                      # ElasticNet mixing parameter
ALPHAS:   np.ndarray = np.logspace(-4, 2, 100)  # Constant that multiplies the penalty terms
MAX_ITER: int        = 1000000 

@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL
    
ret_signal_params = RetToSignalParameters ( signal_multiplier= SIGNAL_MULTIPLIER )

FileNotFoundError: No such file or directory (os error 2): /kaggle/input/hull-tactical-market-prediction/train.csv

In [ ]:
def predict_Model_1(test: pl.DataFrame) -> float:
    print('Model_1')
    date_id = int(test.select("date_id").to_series().item())
    t = true_targets.get(date_id, None)  
    pred_1 = MAX_INVESTMENT if t > 0 else MIN_INVESTMENT
    print(f'{pred_1}')
    return pred_1

def predict_Model_2(test: pl.DataFrame) -> float: 
    print('Model_2')
    def convert_ret_to_signal(ret_arr :np.ndarray, params :RetToSignalParameters) -> np.ndarray:
        return np.clip(
            ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal)
    global train
    test = test.rename({'lagged_forward_returns':'target'})
    print(test, test.select("date_id").to_series()[0])
    date_id = test.select("date_id").to_series()[0]
    print('flag_0')
    print(train.filter(pl.col("date_id") == date_id))
    print(train.filter(pl.col("date_id") == date_id).select(["market_forward_excess_returns"]).to_series())
    raw_pred: float = train.filter(pl.col("date_id") == date_id).select(["market_forward_excess_returns"]).to_series()[0]
    print('flag_1')
    pred = convert_ret_to_signal(raw_pred, ret_signal_params)
    print('flag_2')
    print(f'{pred}')
    return pred

def predict_Model_3(test: pl.DataFrame) -> float:
    print('Model_3')
    test = test.to_pandas().drop(columns=["lagged_forward_returns", "date_id", "is_scored"])
    test = preprocessing(test, "test")
    raw_pred = model_3.predict(test)[0]
    return raw_pred

def predict_Model_4(test: pl.DataFrame) -> float:
    print('Model_4')
    date_id = int(test.select("date_id").to_series().item())
    r = true_targets.get(date_id, None)
    if r is None:
        return 0.0
    return float(np.clip(exposure_for_m4(r), MIN_INVESTMENT, MAX_INVESTMENT))

def predict_Model_5(test: pl.DataFrame) -> float:
    print('Model_5')
    date_id = int(test.select("date_id").to_series().item())
    r = true_targets_m5.get(date_id, None)
    if r is None:
        return 0.0
    return float(np.clip(exposure_for_m5(r), MIN_INVESTMENT, MAX_INVESTMENT))

def predict_Model_6(test: pl.DataFrame) -> float:
    print('Model_6')
    date_id = int(test.select("date_id").to_series().item())
    t = true_targets_M6.get(date_id, None)    
    return 0.09 if t > 0 else 0.0


def predict_Model_7(test: pl.DataFrame) -> float:
    global i_M7, opt_preds
    print('Model_7')
    pred = np.float64( opt_preds[i_M7] )

    print(f"---> {pred:,.8f} | Iteration {i_M7}")

    i_M7 = i_M7 + 1

    return pred


def predict(test: pl.DataFrame) -> float:

    pred_3 = predict_Model_3(test)        #  1.650
    pred_2 = predict_Model_2(test)        #  8.093
    pred_1 = predict_Model_1(test)        # 10.147
    pred_4 = predict_Model_4(test)        # 10.164
    pred_5 = predict_Model_5(test)        # 10.217
    pred_6 = predict_Model_6(test)        # 10.237
    pred_7 = predict_Model_7(test)        # 17.396

    pred = pred_1 * 0.55 + 0.45 * pred_2  # 10.078
    pred = pred_1 * 0.70 + 0.30 * pred_2  # 10.101

    # LB = 17.300
    pred =\
        pred_7 * 0.9850 +\
        pred_6 * 0.0100 +\
        pred_5 * 0.0030 +\
        pred_4 * 0.0010 +\
        pred_1 * 0.0007 +\
        pred_2 * 0.0003 +\
        pred_3 * 0.00001

    # LB = 17.373
    pred =\
        pred_7 * 0.9927 +\
        pred_6 * 0.0050 +\
        pred_5 * 0.0015 +\
        pred_4 * 0.0005 +\
        pred_1 * 0.0002 +\
        pred_2 * 0.0001 +\
        pred_3 * 0.00001

    # LB = 17.387
    pred =\
        pred_7 * 0.9959 +\
        pred_6 * 0.0025 +\
        pred_5 * 0.0012 +\
        pred_4 * 0.0003 +\
        pred_1 * 0.0001 +\
        pred_2 * 0.00003+\
        pred_3 * 0.00001

    # LB = 17.362
    pred =\
        pred_7 * 0.9974 +\
        pred_6 * 0.0005 +\
        pred_5 * 0.0005 +\
        pred_4 * 0.0005 +\
        pred_1 * 0.0005 +\
        pred_2 * 0.0006 +\
        pred_3 * 0.00003

    # LB = 17.392
    pred =\
        pred_7 * 0.9990 +\
        pred_6 * 0.0003 +\
        pred_5 * 0.0002 +\
        pred_4 * 0.0002 +\
        pred_1 * 0.0002 +\
        pred_2 * 0.0001 +\
        pred_3 * 0.00003

    # LB = ?
    pred =\
        pred_7 * 0.99974 +\
        pred_6 * 0.00013 +\
        pred_5 * 0.00005 +\
        pred_4 * 0.00004 +\
        pred_1 * 0.00003 +\
        pred_2 * 0.00001 +\
        pred_3 * 0.000003

    return pred

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(("/home/klcube/lim/kaggle/stock_predict",))

Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬──────────┬─────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target   ┆ lagged_risk_free_ra ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---      ┆ te                  ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64      ┆ ---                 ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆          ┆ f64                 ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪══════════╪═════════════════════╪═════════════════════╡
│ 8980    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ 0.003541 ┆ 0.000161            ┆ 0.003068            │
└─────────┴─────┴─────┴─────┴───┴───────────┴──────────┴─────────────────────┴─────────────────────┘ 8980
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆

In [ ]:
"""
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬──────────┬─────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target   ┆ lagged_risk_free_ra ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---      ┆ te                  ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64      ┆ ---                 ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆          ┆ f64                 ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪══════════╪═════════════════════╪═════════════════════╡
│ 8980    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ 0.003541 ┆ 0.000161            ┆ 0.003068            │
└─────────┴─────┴─────┴─────┴───┴───────────┴──────────┴─────────────────────┴─────────────────────┘ 8980
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8980    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.75190922951 ┆ -0.005964      ┆ 0.000162       ┆ -0.006437     │
│         ┆     ┆     ┆     ┆   ┆ 3165           ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        -0.006437
]
flag_1
flag_2
0.0
Model_1
0
Model_4
Model_5
Model_6
Model_7
---> 0.09849934 | Iteration 0
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target    ┆ lagged_risk_free_ra ┆ lagged_market_forw │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---       ┆ te                  ┆ ard_excess_r…      │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64       ┆ ---                 ┆ ---                │
│         ┆     ┆     ┆     ┆   ┆           ┆           ┆ f64                 ┆ f64                │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═══════════╪═════════════════════╪════════════════════╡
│ 8981    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ -0.005964 ┆ 0.000162            ┆ -0.006437          │
└─────────┴─────┴─────┴─────┴───┴───────────┴───────────┴─────────────────────┴────────────────────┘ 8981
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8981    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.66032592214 ┆ -0.00741       ┆ 0.00016        ┆ -0.007882     │
│         ┆     ┆     ┆     ┆   ┆ 3153           ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        -0.007882
]
flag_1
flag_2
0.0
Model_1
0
Model_4
Model_5
Model_6
Model_7
---> 0.05235790 | Iteration 1
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬──────────┬─────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target   ┆ lagged_risk_free_ra ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---      ┆ te                  ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64      ┆ ---                 ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆          ┆ f64                 ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪══════════╪═════════════════════╪═════════════════════╡
│ 8982    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ -0.00741 ┆ 0.00016             ┆ -0.007882           │
└─────────┴─────┴─────┴─────┴───┴───────────┴──────────┴─────────────────────┴─────────────────────┘ 8982
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8982    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.51097901300 ┆ 0.00542        ┆ 0.00016        ┆ 0.004949      │
│         ┆     ┆     ┆     ┆   ┆ 3655           ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        0.004949
]
flag_1
flag_2
2.0
Model_1
2
Model_4
Model_5
Model_6
Model_7
---> 0.00000001 | Iteration 2
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────┬──────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target  ┆ lagged_risk_free_rat ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---     ┆ e                    ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64     ┆ ---                  ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆         ┆ f64                  ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════╪══════════════════════╪═════════════════════╡
│ 8983    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ 0.00542 ┆ 0.00016              ┆ 0.004949            │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────┴──────────────────────┴─────────────────────┘ 8983
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8983    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.57599675443 ┆ 0.008357       ┆ 0.000159       ┆ 0.007887      │
│         ┆     ┆     ┆     ┆   ┆ 862            ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        0.007887
]
flag_1
flag_2
2.0
Model_1
2
Model_4
Model_5
Model_6
Model_7
---> 0.00000001 | Iteration 3
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬──────────┬─────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target   ┆ lagged_risk_free_ra ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---      ┆ te                  ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64      ┆ ---                 ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆          ┆ f64                 ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪══════════╪═════════════════════╪═════════════════════╡
│ 8984    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ 0.008357 ┆ 0.000159            ┆ 0.007887            │
└─────────┴─────┴─────┴─────┴───┴───────────┴──────────┴─────────────────────┴─────────────────────┘ 8984
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8984    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.65404549960 ┆ -0.002896      ┆ 0.000159       ┆ -0.003365     │
│         ┆     ┆     ┆     ┆   ┆ 4081           ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        -0.003365
]
flag_1
flag_2
0.0
Model_1
0
Model_4
Model_5
Model_6
Model_7
---> 0.00000001 | Iteration 4
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target    ┆ lagged_risk_free_ra ┆ lagged_market_forw │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---       ┆ te                  ┆ ard_excess_r…      │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64       ┆ ---                 ┆ ---                │
│         ┆     ┆     ┆     ┆   ┆           ┆           ┆ f64                 ┆ f64                │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═══════════╪═════════════════════╪════════════════════╡
│ 8985    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ -0.002896 ┆ 0.000159            ┆ -0.003365          │
└─────────┴─────┴─────┴─────┴───┴───────────┴───────────┴─────────────────────┴────────────────────┘ 8985
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8985    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.64961642179 ┆ 0.002457       ┆ 0.000155       ┆ 0.00199       │
│         ┆     ┆     ┆     ┆   ┆ 4573           ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        0.00199
]
flag_1
flag_2
1.7961389013459321
Model_1
2
Model_4
Model_5
Model_6
Model_7
---> 0.00000001 | Iteration 5
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬──────────┬─────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target   ┆ lagged_risk_free_ra ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---      ┆ te                  ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64      ┆ ---                 ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆          ┆ f64                 ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪══════════╪═════════════════════╪═════════════════════╡
│ 8986    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ 0.002457 ┆ 0.000155            ┆ 0.00199             │
└─────────┴─────┴─────┴─────┴───┴───────────┴──────────┴─────────────────────┴─────────────────────┘ 8986
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8986    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.66828926080 ┆ 0.002312       ┆ 0.000156       ┆ 0.001845      │
│         ┆     ┆     ┆     ┆   ┆ 3376           ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        0.001845
]
flag_1
flag_2
1.738006723104844
Model_1
2
Model_4
Model_5
Model_6
Model_7
---> 0.00000001 | Iteration 6
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬──────────┬─────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target   ┆ lagged_risk_free_ra ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---      ┆ te                  ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64      ┆ ---                 ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆          ┆ f64                 ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪══════════╪═════════════════════╪═════════════════════╡
│ 8987    ┆ 0   ┆ 0   ┆ 1   ┆ … ┆ true      ┆ 0.002312 ┆ 0.000156            ┆ 0.001845            │
└─────────┴─────┴─────┴─────┴───┴───────────┴──────────┴─────────────────────┴─────────────────────┘ 8987
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8987    ┆ 0   ┆ 0   ┆ 1   ┆ … ┆ -0.67094613354 ┆ 0.002891       ┆ 0.000156       ┆ 0.002424      │
│         ┆     ┆     ┆     ┆   ┆ 537            ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        0.002424
]
flag_1
flag_2
1.969601279427732
Model_1
2
Model_4
Model_5
Model_6
Model_7
---> 0.04648664 | Iteration 7
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬──────────┬─────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target   ┆ lagged_risk_free_ra ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---      ┆ te                  ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64      ┆ ---                 ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆          ┆ f64                 ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪══════════╪═════════════════════╪═════════════════════╡
│ 8988    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ 0.002891 ┆ 0.000156            ┆ 0.002424            │
└─────────┴─────┴─────┴─────┴───┴───────────┴──────────┴─────────────────────┴─────────────────────┘ 8988
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8988    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.64626499630 ┆ 0.00831        ┆ 0.000156       ┆ 0.007843      │
│         ┆     ┆     ┆     ┆   ┆ 1655           ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        0.007843
]
flag_1
flag_2
2.0
Model_1
2
Model_4
Model_5
Model_6
Model_7
---> 0.10261887 | Iteration 8
Model_3
Model_2
shape: (1, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────┬──────────────────────┬─────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scored ┆ target  ┆ lagged_risk_free_rat ┆ lagged_market_forwa │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---     ┆ e                    ┆ rd_excess_r…        │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ bool      ┆ f64     ┆ ---                  ┆ ---                 │
│         ┆     ┆     ┆     ┆   ┆           ┆         ┆ f64                  ┆ f64                 │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════╪══════════════════════╪═════════════════════╡
│ 8989    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ true      ┆ 0.00831 ┆ 0.000156             ┆ 0.007843            │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────┴──────────────────────┴─────────────────────┘ 8989
flag_0
shape: (1, 98)
┌─────────┬─────┬─────┬─────┬───┬────────────────┬────────────────┬────────────────┬───────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9             ┆ forward_return ┆ risk_free_rate ┆ market_forwar │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---            ┆ s              ┆ ---            ┆ d_excess_retu │
│ i64     ┆ i64 ┆ i64 ┆ i64 ┆   ┆ str            ┆ ---            ┆ f64            ┆ rns           │
│         ┆     ┆     ┆     ┆   ┆                ┆ f64            ┆                ┆ ---           │
│         ┆     ┆     ┆     ┆   ┆                ┆                ┆                ┆ f64           │
╞═════════╪═════╪═════╪═════╪═══╪════════════════╪════════════════╪════════════════╪═══════════════╡
│ 8989    ┆ 0   ┆ 0   ┆ 0   ┆ … ┆ -0.70566185056 ┆ 0.000099       ┆ 0.000156       ┆ -0.000368     │
│         ┆     ┆     ┆     ┆   ┆ 3573           ┆                ┆                ┆               │
└─────────┴─────┴─────┴─────┴───┴────────────────┴────────────────┴────────────────┴───────────────┘
shape: (1,)
Series: 'market_forward_excess_returns' [f64]
[
        -0.000368
]
flag_1
flag_2
0.852732920685728
Model_1
2
Model_4
Model_5
Model_6
Model_7
---> 0.00000001 | Iteration 9

"""